# 01 · KG modes — auto-built knowledge graph + dual-level retrieval

LightRAG extracts a knowledge graph from raw text (entities + relationships, by LLM) and answers with **dual-level retrieval**. This notebook reads the `lightrag_wiki` graph built by `build.py` and runs the same question through all five query modes.

> Run `build.py` first.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
rag = build_rag("lightrag_wiki")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("entities in graph:", f"{len(await g.get_all_labels()):,}")
print("most-connected:", ", ".join(await g.get_popular_labels(limit=10)))

entities in graph: 15,481
most-connected: Armenia, United States, Andhra Pradesh, Abydos, Angola, Azerbaijan, Philip Anthony Hopkins, Alabama, Andrea Andreani, Antonio Agliardi


## The five modes

`naive` = chunks only (baseline); `local` = entity-centric (low-level keywords); `global` = relationship/theme (high-level keywords); `hybrid` = local+global; `mix` = hybrid KG + chunks (default).

In [2]:
question = "Tell me about Armenia: its key facts and the people, places, and events associated with it."
for mode in ["naive", "local", "global", "hybrid", "mix"]:
    ans = await rag.aquery(question, QueryParam(mode=mode, enable_rerank=False))
    print(f"\n### {mode}\n" + str(ans).strip()[:500] + " …")


### naive
### Key Facts about Armenia

- **Geography**: Armenia is a landlocked country located in the South Caucasus region, straddling Eastern Europe and West Asia. It borders Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. The terrain is predominantly mountainous with the highest point being Mount Aragats. Armenia's climate is highland continental featuring hot summers and cold winters. Approximately half of the country lies at elevations above 1,000 meters abov …



### local
Armenia, a landlocked country located in the South Caucasus region between Eastern Europe and Western Asia, encompasses a rich tapestry of history, culture, and geography. Here are some key facts about Armenia, alongside notable people, places, and events associated with it.

### Key Facts About Armenia

- **Geography**: Armenia is bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. It features a predominantly mountainous terrain, including its hi …



### global
### Overview of Armenia

Armenia is a landlocked country located in the South Caucasus region of Eurasia, at the crossroads of Europe and Asia. It shares borders with Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. The country features a mountainous terrain and experiences a continental highland climate, known for hot summers and cold winters. The highest point in Armenia is Mount Aragats.

### Key Facts

- **Capital City**: Yerevan is the capital and lar …



### hybrid
### Key Facts about Armenia

- **Location**: Armenia is a landlocked country situated in the South Caucasus region of Eurasia, bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. 
- **Geography**: The country has a predominantly mountainous terrain and experiences a continental highland climate, characterized by hot summers and cold winters. The highest point is Mount Aragats, while significant rivers include the Araks and Debed rivers.
- **Capita …



### mix
### Overview of Armenia

**Geography and Location**  
Armenia is a landlocked country situated in the South Caucasus region of Eurasia, bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. The country is characterized by its mountainous terrain and a continental highland climate, largely dominated by the Lesser Caucasus mountain range. The highest point in Armenia is Mount Aragats.

**Population and Demographics**  
Historically, Armenia's populati …


## The dual-level signal (`aquery_data`)

The retrieved structure makes the difference visible: `local` leans on low-level (entity) keywords, `global` on high-level (theme) keywords, `mix` pulls entities + relationships + raw chunks.

In [3]:
for mode in ["local", "global", "mix"]:
    d = await rag.aquery_data(question, QueryParam(mode=mode, enable_rerank=False))
    data, kw = d.get("data", {}), d.get("metadata", {}).get("keywords", {})
    print(f"{mode:7} entities={len(data.get('entities',[])):3} "
          f"relationships={len(data.get('relationships',[])):3} chunks={len(data.get('chunks',[])):3} "
          f"| high-level={kw.get('high_level')} low-level={kw.get('low_level')}")

local   entities= 39 relationships= 40 chunks= 20 | high-level=['Armenia', 'key facts', 'people', 'places', 'events'] low-level=[]


global  entities= 40 relationships= 40 chunks= 20 | high-level=['Armenia', 'key facts', 'culture', 'people', 'places', 'events'] low-level=['Armenia']


mix     entities= 60 relationships=143 chunks= 20 | high-level=['Armenia', 'key facts', 'people', 'places', 'events'] low-level=['Armenian culture', 'Geography', 'history', 'traditions']


## `only_need_context` — what was retrieved, without an LLM answer

In [4]:
ctx = await rag.aquery(question, QueryParam(mode="mix", only_need_context=True))
print(str(ctx).strip()[:1100])

Knowledge Graph Data (Entity):

```json
{"entity": "Armenia", "type": "location", "description": "Armenia is a landlocked country located in the South Caucasus region of Eurasia, bordered by Georgia to the north, Azerbaijan to the east, Iran to the south, and Turkey to the west. The country has a mountainous terrain and experiences a continental highland climate. Throughout its history, particularly during the Soviet period, Armenia had a less developed industrial sector compared to other nations. However, since gaining independence, it has undergone significant economic growth and transformation, evolving into a primarily industrial and agricultural economy.\n\nArmenia shares a complex relationship with its neighbor Azerbaijan, particularly concerning territorial disputes over regions such as Nagorno-Karabakh. These disputes have led to historical and ongoing conflicts between the two nations, contributing to significant population changes in Armenia. The population peaked at 3.633 mi

## Follow-up questions (`conversation_history`)

LightRAG carries multi-turn context, so a follow-up like *“what countries does it border?”* resolves *“it”* from the previous turn.

In [5]:
history = [
    {"role": "user", "content": "Tell me about Armenia."},
    {"role": "assistant", "content": "Armenia is a landlocked country in the South Caucasus."},
]
follow_up = "What countries does it border?"
ans = await rag.aquery(follow_up, QueryParam(mode="mix", enable_rerank=False, conversation_history=history))
print("Follow-up:", follow_up, "\n")
print(str(ans).strip()[:500])

Follow-up: What countries does it border? 

Armenia is bordered by several countries:

- **Georgia** to the north
- **Azerbaijan** to the east, with which it has complex territorial disputes, particularly over the Nagorno-Karabakh region
- **Iran** to the south
- **Turkey** to the west, where relations are historically tense due to various grievances.

Overall, Armenia's geographical position at the crossroads of Eastern Europe and Western Asia gives it significant strategic importance in the region. 

### References

- [7] Geography of A


## How this graph was built

`build.py` streams Wikipedia and hands batches to `ainsert` — LightRAG chunks each article, extracts entities + relationships with the LLM, and merges them across articles:

```python
rag = build_rag("lightrag_wiki")
await rag.ainsert(texts, ids=ids, file_paths=titles)   # the LLM extracts the KG
```

This notebook reads the finished graph; run `build.py` to (re)build it.

In [6]:
await rag.finalize_storages()